# GPT-2 vs RWKV on Wiki Data — Pareto Efficiency Plots

**Repo:** [ai-transformer-efficiency-comparison](https://github.com/TylrDn/ai-transformer-efficiency-comparison)  
**Phase:** 3 — Notebook Pipelines

Trains and evaluates GPT-2 and RWKV on the wiki JSONL dataset, then plots
Pareto-optimal curves across perplexity, FLOPs, latency, and memory.

### References
- GPT-2: Radford et al. (2019) https://openai.com/research/language-unsupervised
- RWKV: Peng et al. (2023) https://arxiv.org/abs/2305.13048
- HuggingFace Transformers: Wolf et al. (2020) https://arxiv.org/abs/1910.03771

In [ ]:
from __future__ import annotations

import logging
import os
from dataclasses import dataclass, field
from pathlib import Path

import torch
import wandb

logging.basicConfig(level=logging.INFO, format="%(asctime)s — %(levelname)s — %(message)s")
logger = logging.getLogger(__name__)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
WANDB_DISABLED = os.getenv("WANDB_DISABLED", "false").lower() == "true"


@dataclass
class ComparisonConfig:
    """Hyperparameters for the efficiency comparison experiment."""

    wiki_data_path: str = "data/processed/wiki_jsonl"
    models: list[str] = field(default_factory=lambda: ["gpt2", "RWKV/rwkv-4-169m-pile"])
    max_length: int = 512
    eval_samples: int = 1000
    batch_size: int = 8
    output_dir: Path = Path("results/efficiency_comparison")


cfg = ComparisonConfig()
cfg.output_dir.mkdir(parents=True, exist_ok=True)

if not WANDB_DISABLED:
    run = wandb.init(
        project="ai-transformer-efficiency-comparison",
        config={
            "models": cfg.models,
            "max_length": cfg.max_length,
            "eval_samples": cfg.eval_samples,
        },
    )

logger.info("Config: %s", cfg)

In [ ]:
import math
import time

import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer


def load_wiki_eval_dataset(data_path: str, n_samples: int, max_length: int) -> list[str]:
    """Load a sample of the Wiki JSONL dataset for evaluation.

    Assumes shards produced by notebook 01_wiki_preprocessing.
    """
    import glob

    files = sorted(glob.glob(f"{data_path}/shard_*.jsonl"))
    if not files:
        logger.warning("No JSONL shards found — using dummy data for demo")
        return ["The transformer architecture revolutionized NLP."] * n_samples

    ds = load_dataset("json", data_files={"train": files}, split="train")
    texts = ds.select(range(min(n_samples, len(ds))))["text"]
    return [t[:max_length] for t in texts]


texts = load_wiki_eval_dataset(cfg.wiki_data_path, cfg.eval_samples, cfg.max_length * 4)
logger.info("Loaded %d evaluation texts", len(texts))

In [ ]:
from dataclasses import asdict


def evaluate_model(
    model_id: str,
    texts: list[str],
    cfg: ComparisonConfig,
    device: torch.device = DEVICE,
) -> dict[str, float]:
    """Compute perplexity, throughput, latency, and memory for a model.

    Returns a metrics dict compatible with Pareto analysis.
    """
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16)
    model = model.to(device).eval()

    param_count = sum(p.numel() for p in model.parameters()) / 1e6  # millions

    nlls: list[float] = []
    latencies: list[float] = []

    if device.type == "cuda":
        torch.cuda.reset_peak_memory_stats(device)

    for i in range(0, min(len(texts), cfg.eval_samples), cfg.batch_size):
        batch = texts[i : i + cfg.batch_size]
        enc = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            max_length=cfg.max_length,
            padding=True,
        ).to(device)

        t0 = time.perf_counter()
        with torch.no_grad():
            out = model(**enc, labels=enc["input_ids"])
        if device.type == "cuda":
            torch.cuda.synchronize()
        latencies.append(time.perf_counter() - t0)
        nlls.append(out.loss.item())

    ppl = math.exp(sum(nlls) / len(nlls))
    avg_lat_ms = sum(latencies) / len(latencies) * 1000
    peak_mem_gb = (
        torch.cuda.max_memory_allocated(device) / 1e9 if device.type == "cuda" else float("nan")
    )

    del model
    torch.cuda.empty_cache() if device.type == "cuda" else None

    return {
        "model": model_id,
        "params_m": param_count,
        "perplexity": ppl,
        "latency_ms": avg_lat_ms,
        "peak_mem_gb": peak_mem_gb,
    }


model_results: list[dict] = []
for model_id in cfg.models:
    logger.info("Evaluating %s ...", model_id)
    try:
        metrics = evaluate_model(model_id, texts, cfg)
        model_results.append(metrics)
        logger.info("%s → PPL=%.2f  lat=%.1f ms  mem=%.2f GB", model_id,
                    metrics["perplexity"], metrics["latency_ms"], metrics["peak_mem_gb"])
        if not WANDB_DISABLED:
            wandb.log({f"{model_id}/{k}": v for k, v in metrics.items() if k != "model"})
    except Exception as exc:
        logger.error("Failed to evaluate %s: %s", model_id, exc)

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

df = pd.DataFrame(model_results)
df.to_csv(cfg.output_dir / "efficiency_comparison.csv", index=False)

fig, ax = plt.subplots(figsize=(8, 6))
for _, row in df.iterrows():
    ax.scatter(row["latency_ms"], row["perplexity"], s=row["params_m"] * 0.5, alpha=0.8)
    ax.annotate(row["model"].split("/")[-1], (row["latency_ms"], row["perplexity"]),
                fontsize=9, ha="right")

ax.set_xlabel("Average Latency (ms / batch)")
ax.set_ylabel("Perplexity (lower is better)")
ax.set_title("Pareto Efficiency: GPT-2 vs RWKV on Wikipedia Data")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plot_path = cfg.output_dir / "pareto_plot.png"
plt.savefig(plot_path, dpi=150)
plt.show()

if not WANDB_DISABLED:
    wandb.log({"pareto_plot": wandb.Image(str(plot_path))})
    artifact = wandb.Artifact("efficiency-comparison", type="results")
    artifact.add_file(str(cfg.output_dir / "efficiency_comparison.csv"))
    artifact.add_file(str(plot_path))
    wandb.log_artifact(artifact)

df